# Banking Transaction Pipeline Test Suite

Comprehensive test cases for the **Banking Data Engineering Platform** project.

## Project Overview:
- **Catalog**: `banking_catalog`
- **Schema**: `banking_project`
- **Volume**: `banking_volume`
- **Source Systems**: ATM, UPI, NEFT, RTGS, CARD, INTERNET_BANKING, MOBILE_BANKING, BRANCH

## Test Coverage:
1. **Unit Tests** - BankingTestDataGenerator Class
2. **Unit Tests** - Transaction Generation
3. **Unit Tests** - Master Data Generation
4. **Integration Tests** - File Operations (CSV/Excel)
5. **Integration Tests** - Unity Catalog Volume Integration
6. **Data Validation Tests** - Schema & Constraints
7. **End-to-End Tests** - Full Data Generation Pipeline

## Notes:
⚠️ **Important**: These tests require Python/Spark compute. Switch from SQL Warehouse to a cluster with Python support before running.

In [0]:
# Import required libraries
import sys
import os
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Add project path
project_path = '/Workspace/Users/swapniltake1@outlook.com/Automated-Banking-Transaction-Pipeline-Databricks-Free-Edition-'
sys.path.insert(0, f'{project_path}/tests')

print("✅ Libraries imported successfully")
print(f"Project path: {project_path}")

In [0]:
%run ./99_Config

In [0]:
def test_banking_data_generator_initialization():
    """
    Test that BankingTestDataGenerator initializes correctly with proper configuration
    """
    print("Testing BankingTestDataGenerator Initialization...")
    print("="*80)
    
    # Run the TestDataGenerator notebook to load the class
    %run ./TestDataGenerator
    
    # Initialize generator
    generator = BankingTestDataGenerator(
        base_path=BASE_PATH,
        source_systems=SOURCE_SYSTEMS,
        valid_transaction_types=VALID_TRANSACTION_TYPES
    )
    
    # Verify initialization
    assert generator.base_path == BASE_PATH, "Base path mismatch"
    assert len(generator.source_systems) == 8, f"Expected 8 source systems, got {len(generator.source_systems)}"
    assert len(generator.valid_transaction_types) == 8, f"Expected 8 transaction types, got {len(generator.valid_transaction_types)}"
    assert generator.incoming_path == f"{BASE_PATH}/incoming", "Incoming path mismatch"
    assert generator.master_data_path == f"{BASE_PATH}/master_data", "Master data path mismatch"
    
    print("\n✅ Test Passed: BankingTestDataGenerator initialized correctly")
    print(f"  Base Path: {generator.base_path}")
    print(f"  Source Systems: {len(generator.source_systems)}")
    print(f"  Transaction Types: {len(generator.valid_transaction_types)}")
    print(f"  Incoming Path: {generator.incoming_path}")
    print(f"  Master Data Path: {generator.master_data_path}")
    
    return generator

# Run the test
generator = test_banking_data_generator_initialization()

In [0]:
def test_generate_transactions_single_source():
    """
    Test transaction generation for a single source system (ATM)
    """
    print("\nTesting Transaction Generation for ATM...")
    print("="*80)
    
    # Generate ATM transactions
    atm_df = generator.generate_transactions(
        source_system='ATM',
        num_records=100,
        days_back=7
    )
    
    # Verify record count
    assert len(atm_df) == 100, f"Expected 100 records, got {len(atm_df)}"
    
    # Verify required columns exist
    required_columns = [
        'transaction_id', 'customer_id', 'account_number', 'transaction_type',
        'amount', 'currency', 'branch', 'transaction_timestamp', 'atm_id', 'card_number'
    ]
    
    for col in required_columns:
        assert col in atm_df.columns, f"Missing required column: {col}"
    
    # Verify data types
    assert atm_df['amount'].dtype == 'float64', "Amount should be float"
    assert atm_df['transaction_id'].dtype == 'object', "Transaction ID should be string"
    
    # Verify transaction types are valid (should be from VALID_TRANSACTION_TYPES config)
    assert atm_df['transaction_type'].isin(VALID_TRANSACTION_TYPES).all(), "Invalid transaction types found"
    
    # Display sample data
    print("\nSample ATM Transactions:")
    print(atm_df.head(5))
    
    print("\n✅ Test Passed: Transaction generation for ATM works correctly")
    print(f"  Records generated: {len(atm_df)}")
    print(f"  Columns: {len(atm_df.columns)}")
    print(f"  Date range: {atm_df['transaction_timestamp'].min()} to {atm_df['transaction_timestamp'].max()}")
    
test_generate_transactions_single_source()

In [0]:
def test_generate_transactions_all_sources():
    """
    Test transaction generation for all source systems
    """
    print("\nTesting Transaction Generation for All Source Systems...")
    print("="*80)
    
    test_results = []
    expected_sources = ['ATM', 'UPI', 'NEFT', 'RTGS', 'CARD', 'INTERNET_BANKING', 'MOBILE_BANKING', 'BRANCH']
    
    for source in expected_sources:
        try:
            df = generator.generate_transactions(
                source_system=source,
                num_records=50,
                days_back=7
            )
            
            # Verify record count
            assert len(df) == 50, f"{source}: Expected 50 records, got {len(df)}"
            
            test_results.append({'source': source, 'records': len(df), 'status': '✅ Pass'})
            print(f"  {source:20} - Generated {len(df)} records")
            
        except Exception as e:
            test_results.append({'source': source, 'records': 0, 'status': f'❌ Fail: {str(e)}'})
            print(f"  {source:20} - ❌ Failed: {str(e)}")
    
    # Verify all sources were tested
    assert len(test_results) == len(expected_sources), "Not all source systems were tested"
    
    # Verify all tests passed
    failed_tests = [r for r in test_results if '❌' in r['status']]
    assert len(failed_tests) == 0, f"Some tests failed: {failed_tests}"
    
    print("\n✅ Test Passed: Transaction generation works for all source systems")
    print(f"  Total source systems tested: {len(test_results)}")
    print(f"  All tests passed: {len([r for r in test_results if '✅' in r['status']])}")
    
test_generate_transactions_all_sources()

In [0]:
def test_generate_master_data_customers():
    """
    Test customer master data generation
    """
    print("\nTesting Customer Master Data Generation...")
    print("="*80)
    
    # Generate customer data
    customers_df = generator.generate_customers(num_records=100)
    
    # Verify record count
    assert len(customers_df) == 100, f"Expected 100 customers, got {len(customers_df)}"
    
    # Verify required columns
    required_columns = [
        'customer_id', 'full_name', 'email', 'phone', 
        'city', 'state', 'customer_segment', 
        'registration_date', 'kyc_status'
    ]
    
    for col in required_columns:
        assert col in customers_df.columns, f"Missing required column: {col}"
    
    # Verify customer_id format (should be CUSTXXXXX)
    sample_cust_id = customers_df['customer_id'].iloc[0]
    assert sample_cust_id.startswith('CUST'), "Customer ID should start with 'CUST'"
    
    # Verify customer segments are valid
    valid_segments = ['RETAIL', 'PREMIUM', 'SME', 'CORPORATE']
    assert customers_df['customer_segment'].isin(valid_segments).all(), "Invalid customer segments found"
    
    # Verify KYC status values
    valid_kyc = ['COMPLETED', 'PENDING', 'IN_PROGRESS']
    assert customers_df['kyc_status'].isin(valid_kyc).all(), "Invalid KYC status found"
    
    # Display sample data
    print("\nSample Customer Records:")
    print(customers_df.head(5))
    
    print("\n✅ Test Passed: Customer master data generation works correctly")
    print(f"  Records generated: {len(customers_df)}")
    print(f"  Segments: {customers_df['customer_segment'].value_counts().to_dict()}")
    print(f"  KYC Status: {customers_df['kyc_status'].value_counts().to_dict()}")
    
test_generate_master_data_customers()

In [0]:
def test_generate_master_data_accounts():
    """
    Test account master data generation
    """
    print("\nTesting Account Master Data Generation...")
    print("="*80)
    
    # Generate account data
    accounts_df = generator.generate_accounts(num_records=150)
    
    # Verify record count
    assert len(accounts_df) == 150, f"Expected 150 accounts, got {len(accounts_df)}"
    
    # Verify required columns
    required_columns = [
        'account_number', 'customer_id', 'account_type', 'current_balance',
        'currency', 'branch_code', 'opening_date', 'status'
    ]
    
    for col in required_columns:
        assert col in accounts_df.columns, f"Missing required column: {col}"
    
    # Verify account_number format (should be 12 digits)
    sample_acc = accounts_df['account_number'].iloc[0]
    assert len(str(sample_acc)) == 12, "Account number should be 12 digits"
    
    # Verify account types are valid
    valid_types = ['SAVINGS', 'CURRENT', 'FIXED_DEPOSIT', 'SALARY']
    assert accounts_df['account_type'].isin(valid_types).all(), "Invalid account types found"
    
    # Verify account status values
    valid_status = ['ACTIVE', 'INACTIVE', 'DORMANT', 'CLOSED']
    assert accounts_df['status'].isin(valid_status).all(), "Invalid account status found"
    
    # Verify balance is positive
    assert (accounts_df['current_balance'] >= 0).all(), "Balance should be non-negative"
    
    # Display sample data
    print("\nSample Account Records:")
    print(accounts_df.head(5))
    
    print("\n✅ Test Passed: Account master data generation works correctly")
    print(f"  Records generated: {len(accounts_df)}")
    print(f"  Account Types: {accounts_df['account_type'].value_counts().to_dict()}")
    print(f"  Status: {accounts_df['status'].value_counts().to_dict()}")
    print(f"  Average Balance: {accounts_df['current_balance'].mean():.2f}")
    
test_generate_master_data_accounts()

In [0]:
def test_generate_master_data_branches():
    """
    Test branch master data generation
    """
    print("\nTesting Branch Master Data Generation...")
    print("="*80)
    
    # Generate branch data
    branches_df = generator.generate_branches(num_records=50)
    
    # Verify record count
    assert len(branches_df) == 50, f"Expected 50 branches, got {len(branches_df)}"
    
    # Verify required columns
    required_columns = [
        'branch_code', 'branch_name', 'address', 'city', 
        'state', 'ifsc_code', 'phone'
    ]
    
    for col in required_columns:
        assert col in branches_df.columns, f"Missing required column: {col}"
    
    # Verify branch_code format (should be BRXXX)
    sample_branch = branches_df['branch_code'].iloc[0]
    assert sample_branch.startswith('BR'), "Branch code should start with 'BR'"
    
    # Verify IFSC code format (should be 11 characters)
    sample_ifsc = branches_df['ifsc_code'].iloc[0]
    assert len(sample_ifsc) == 11, "IFSC code should be 11 characters"
    
    # Display sample data
    print("\nSample Branch Records:")
    print(branches_df.head(5))
    
    print("\n✅ Test Passed: Branch master data generation works correctly")
    print(f"  Records generated: {len(branches_df)}")
    print(f"  Cities: {branches_df['city'].nunique()} unique cities")
    print(f"  States: {branches_df['state'].nunique()} unique states")
    
test_generate_master_data_branches()

In [0]:
def test_transaction_amount_validation():
    """
    Test that transaction amounts are within valid range
    """
    print("\nTesting Transaction Amount Validation...")
    print("="*80)
    
    # Generate transaction data
    test_df = generator.generate_transactions(
        source_system='CARD',
        num_records=500,
        days_back=7
    )
    
    # Verify amounts are positive
    assert (test_df['amount'] > 0).all(), "All amounts should be positive"
    
    # Verify amounts are within reasonable range (based on config)
    min_amount = MIN_TRANSACTION_AMOUNT
    max_amount = MAX_TRANSACTION_AMOUNT
    
    assert (test_df['amount'] >= min_amount).all(), f"All amounts should be >= {min_amount}"
    assert (test_df['amount'] <= max_amount).all(), f"All amounts should be <= {max_amount}"
    
    # Calculate statistics
    avg_amount = test_df['amount'].mean()
    median_amount = test_df['amount'].median()
    min_found = test_df['amount'].min()
    max_found = test_df['amount'].max()
    
    print("\nTransaction Amount Statistics:")
    print(f"  Count: {len(test_df)}")
    print(f"  Min: {min_found:,.2f}")
    print(f"  Max: {max_found:,.2f}")
    print(f"  Average: {avg_amount:,.2f}")
    print(f"  Median: {median_amount:,.2f}")
    
    print("\n✅ Test Passed: Transaction amounts are within valid range")
    
test_transaction_amount_validation()

In [0]:
def test_no_null_values_in_critical_columns():
    """
    Test that critical columns don't contain null values
    """
    print("\nTesting Null Values in Critical Columns...")
    print("="*80)
    
    # Generate test data
    test_df = generator.generate_transactions(
        source_system='UPI',
        num_records=200,
        days_back=7
    )
    
    # Define critical columns that should not have nulls
    critical_columns = [
        'transaction_id', 'customer_id', 'account_number',
        'transaction_type', 'amount', 'currency'
    ]
    
    null_counts = {}
    for col in critical_columns:
        null_count = test_df[col].isna().sum()
        null_counts[col] = null_count
        assert null_count == 0, f"Column '{col}' has {null_count} null values"
    
    print("\nNull Value Check Results:")
    for col, count in null_counts.items():
        print(f"  {col:20} - {count} nulls (✅ Pass)")
    
    print("\n✅ Test Passed: No null values in critical columns")
    
test_no_null_values_in_critical_columns()

In [0]:
def test_csv_file_operations():
    """
    Test CSV file saving and verification
    """
    print("\nTesting CSV File Operations...")
    print("="*80)
    
    # Generate test transaction data
    test_df = generator.generate_transactions(
        source_system='NEFT',
        num_records=100,
        days_back=7
    )
    
    # Save to CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    test_file_path = f"{TEMP_PATH}/test_transactions_{timestamp}"
    
    try:
        saved_file = generator.save_to_csv(test_df, test_file_path)
        
        # Verify file was created
        assert saved_file is not None, "File path should be returned"
        print(f"\n✅ File created successfully: {saved_file}")
        
        # Verify file exists
        file_exists = len(dbutils.fs.ls(os.path.dirname(saved_file))) > 0
        assert file_exists, "File should exist in the file system"
        
        print("\n✅ Test Passed: CSV file operations work correctly")
        print(f"  File location: {saved_file}")
        print(f"  Records saved: {len(test_df)}")
        
    except Exception as e:
        print(f"\n❌ Test Failed: {str(e)}")
        raise
    
test_csv_file_operations()

In [0]:
def test_bulk_transaction_generation():
    """
    Test bulk generation of transaction files for all source systems
    """
    print("\nTesting Bulk Transaction File Generation...")
    print("="*80)
    
    try:
        # Generate transaction files for all source systems
        transaction_files = generator.generate_all_transaction_files(
            records_per_source=100,
            format='csv'
        )
        
        # Verify all source systems have files generated
        expected_count = len(SOURCE_SYSTEMS)
        actual_count = len(transaction_files)
        
        assert actual_count == expected_count, f"Expected {expected_count} files, got {actual_count}"
        
        # Verify file info structure
        for file_info in transaction_files:
            assert 'source_system' in file_info, "File info should contain source_system"
            assert 'record_count' in file_info, "File info should contain record_count"
            assert 'file_path' in file_info, "File info should contain file_path"
            assert file_info['record_count'] == 100, f"Expected 100 records, got {file_info['record_count']}"
        
        # Calculate total records
        from builtins import sum as py_sum
        total_records = py_sum(f['record_count'] for f in transaction_files)
        
        print("\nBulk Generation Summary:")
        print(f"  Total source systems: {actual_count}")
        print(f"  Total records generated: {total_records:,}")
        print(f"  Records per source: {transaction_files[0]['record_count']}")
        
        print("\n✅ Test Passed: Bulk transaction file generation works correctly")
        
    except Exception as e:
        print(f"\n❌ Test Failed: {str(e)}")
        raise
    
test_bulk_transaction_generation()

## Test Summary

All test cases have been implemented for the Banking Data Engineering Platform:

### Unit Tests - BankingTestDataGenerator
1. ✅ Class initialization with proper configuration
2. ✅ Transaction generation for single source system (ATM)
3. ✅ Transaction generation for all source systems (8 systems)
4. ✅ Customer master data generation
5. ✅ Account master data generation
6. ✅ Branch master data generation

### Data Validation Tests
7. ✅ Transaction amount range validation
8. ✅ Null values check in critical columns

### Integration Tests
9. ✅ CSV file save and verify operations
10. ✅ End-to-end bulk transaction file generation

---

### Project Configuration:
- **Catalog**: `banking_catalog`
- **Schema**: `banking_project`
- **Volume**: `banking_volume`
- **Source Systems**: ATM, UPI, NEFT, RTGS, CARD, INTERNET_BANKING, MOBILE_BANKING, BRANCH
- **Transaction Types**: ATM_WITHDRAWAL, UPI_TRANSFER, NEFT, RTGS, CARD_PAYMENT, IB_TRANSFER, MB_TRANSFER, CASH_DEPOSIT

### Next Steps:
- Run these tests on a **Python/Spark compute cluster** (not SQL Warehouse)
- Add Bronze/Silver/Gold layer transformation tests
- Add streaming ingestion tests
- Create pytest files for CI/CD integration
- Add data quality monitoring tests